# Chapter 34
## Nested Gamma Theta Rhythms
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter34.ipynb)

## About this chapter

These examples nest gamma-band PING episodes inside a slower theta cycle.
They first build up the O-LM (oriens-lacunosum moleculare) interneuron
model piece by piece -- a pre-O-LM cell without slow currents, then the
h-current alone, then h- and A-currents together -- and then combine E, I,
and O-LM populations, either as a full three-population network or as a
two-population E-I (PING) network whose drive or inhibition is itself
modulated at theta frequency.

Theta modulation opens and closes windows for gamma: whether it acts through
a periodic boost to the E-cell drive or a periodic pulse of external
inhibition, only part of each theta cycle supports E-I spiking. The O-LM
cell's slow currents shape how it paces this window from the inhibitory
side -- h-current promotes a slow post-inhibitory rebound, while A-current
delays the OLM cell's own spiking, and together they set the timing of the
OLM-driven inhibition that gates the E and I populations.

The conductance-based currents throughout keep the familiar form
$I = g\,s\,(v - E_{\rm rev})$, with `g_h`, `g_A` controlling the OLM cell's
extra slow currents and network coupling strengths `g_hat_XY` (together with
connection probabilities `p_XY`) setting the expected total conductance
between populations, normalized by expected in-degree as in earlier PING/ING
chapters. Nesting is read from population rasters and mean-voltage/mean
-synaptic-gate (LFP-like) traces plotted against the slower theta rhythm.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

## Pre-O-LM/O-LM cell model and shared network helpers

Every example below builds on a common gating-variable/model layer:

- `alpha_h`/`alpha_m`/`alpha_n`/`beta_h`/`beta_m`/`beta_n` and their
  steady-state/time-constant pairs `h_inf`/`m_inf`/`n_inf`,
  `tau_h`/`tau_m`/`tau_n` are the pre-O-LM cell's fast gates -- shared,
  unchanged, by the O-LM cell itself (a pre-O-LM cell plus h- and
  A-currents).
- `r_inf`/`tau_r` (h-current) and `a_inf`/`b_inf`/`tau_a`/`tau_b`
  (A-current) are the O-LM cell's two slow gates.
- `m_e_inf`/`h_e_inf`/`tau_h_e`/`n_e_inf`/`tau_n_e` (RTM E-cell) and
  `m_i_inf`/`h_i_inf`/`tau_h_i`/`n_i_inf`/`tau_n_i` (WB I-cell) are the same
  pyramidal/interneuron gates used in Chapters 30-33.
- `tau_peak_function`/`tau_d_q_function` solve for the double-exponential
  synapse's rise-time constant, and `rtm_init_population`/
  `wb_init_population`/`olm_init_population` splay-initialize a population
  of E-, I-, or O-LM cells onto their limit cycle (or resting point), exactly
  as in `rtmInit`/`wbInit`/`olmInit` from the original `lib.py` files.

Two network simulators are built on top of this: `simulate_eio_network`
(E-I-O-LM, `EIO_1`) integrates with `scipy.integrate.odeint`, matching the
original script -- its E-cell synaptic term uses the same sign convention as
the original `lib.py` (noted in the docstring), which makes an explicit
fixed-step integrator diverge, so `odeint`'s adaptive stepping is kept
rather than a numba port. `simulate_ping_theta_drive`/
`simulate_ping_theta_inhibition` (PING with theta drive/inhibition) use a
numba-accelerated fixed-step network stepper, following the same pattern as
Chapters 30-31's PING/ING network steppers.

In [ ]:
import math
import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from numba import njit
from numba.typed import List
from ipywidgets import interact


# ---------------------------------------------------------------- pre-OLM / OLM gating

def alpha_h(v):
    return 0.07 * exp(-(v + 63.0) / 20.0)


def alpha_m(v):
    q = (v + 38.0) / 10.0
    return q / (1.0 - exp(-q))


def alpha_n(v):
    return 0.018 * (v - 25.0) / (1.0 - exp(-(v - 25.0) / 25.0))


def beta_h(v):
    return 1.0 / (exp(-(v + 33.0) / 10.0) + 1.0)


def beta_m(v):
    return 4.0 * exp(-(v + 65.0) / 18.0)


def beta_n(v):
    return 0.0036 * (35.0 - v) / (1.0 - exp(-(35.0 - v) / 12.0))


def h_inf(v):
    return alpha_h(v) / (alpha_h(v) + beta_h(v))


def m_inf(v):
    return alpha_m(v) / (alpha_m(v) + beta_m(v))


def n_inf(v):
    return alpha_n(v) / (alpha_n(v) + beta_n(v))


def tau_h(v):
    return 1.0 / (alpha_h(v) + beta_h(v))


def tau_m(v):
    return 1.0 / (alpha_m(v) + beta_m(v))


def tau_n(v):
    return 1.0 / (alpha_n(v) + beta_n(v))


def r_inf(v):
    return 1.0 / (1.0 + exp((v + 84.0) / 10.2))


def tau_r(v):
    return 1.0 / (exp(-14.59 - 0.086 * v) + exp(-1.87 + 0.0701 * v))


def a_inf(v):
    return 1.0 / (1.0 + exp(-(v + 14.0) / 16.6))


def b_inf(v):
    return 1.0 / (1.0 + exp((v + 71.0) / 7.3))


def tau_a(v):
    return 5.0


def tau_b(v):
    return 1.0 / (0.000009 / exp((v - 26.0) / 28.5) +
                  0.014 / (0.2 + exp(-(v + 70.0) / 11.0)))


def derivative_pre_olm(x0, t, i_ext):
    v, h, n = x0
    dv = (i_ext - 30.0 * h * m_inf(v) ** 3 * (v - 90.0)
          - 23.0 * n ** 4 * (v + 100.0) - 0.05 * (v + 70.0)) / 1.3
    dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
    dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
    return [dv, dh, dn]


def simulate_pre_olm_voltage_trace(i_ext=1.5, t_final=200.0, dt=0.01):
    v0 = -63.0
    x0 = [v0, h_inf(v0), n_inf(v0)]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative_pre_olm, x0, t, args=(i_ext,))
    return t, sol[:, 0]


def derivative_olm_h(x0, t, i_ext, g_h):
    v, h, n, r = x0
    dv = (i_ext - 30.0 * h * m_inf(v) ** 3 * (v - 90.0)
          - 23.0 * n ** 4 * (v + 100.0) - 0.05 * (v + 70.0)
          - g_h * r * (v + 32.9)) / 1.3
    dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
    dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
    dr = (r_inf(v) - r) / tau_r(v)
    return [dv, dh, dn, dr]


def simulate_olm_h_current(i_ext=0.0, g_h=12.0, t_final=200.0, dt=0.01):
    v0 = -63.0
    x0 = [v0, h_inf(v0), n_inf(v0), r_inf(v0)]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative_olm_h, x0, t, args=(i_ext, g_h))
    return t, sol[:, 0], sol[:, 3]


def derivative_olm_h_and_a(x0, t, i_ext, g_h, g_A):
    v, h, n, r, a, b = x0
    I_K = 23.0 * n ** 4 * (v + 100.0)
    I_Na = 30.0 * h * m_inf(v) ** 3 * (v - 90.0)
    I_L = 0.05 * (v + 70.0)
    I_H = g_h * r * (v + 32.9)
    I_A = g_A * a * b * (v + 90.0)
    dv = (i_ext - I_L - I_K - I_Na - I_H - I_A) / 1.3
    dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
    dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
    dr = (r_inf(v) - r) / tau_r(v)
    da = (a_inf(v) - a) / tau_a(v)
    db = (b_inf(v) - b) / tau_b(v)
    return [dv, dh, dn, dr, da, db]


def simulate_olm_h_and_a_currents(i_ext=0.0, g_h=12.0, g_A=22.0, t_final=500.0, dt=0.01):
    v0 = -63.0
    x0 = [v0, h_inf(v0), n_inf(v0), r_inf(v0), a_inf(v0), b_inf(v0)]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative_olm_h_and_a, x0, t, args=(i_ext, g_h, g_A))
    return t, sol[:, 0], sol[:, 4], sol[:, 5]


# ---------------------------------------------------------------- static gate plots

def plot_pre_olm_gates():
    v = np.arange(-100, 50, 0.01)
    fig, ax = plt.subplots(nrows=3, ncols=2, figsize=(6, 5.5), sharex=True)
    ax[0, 0].plot(v, m_inf(v), color='k', lw=2)
    ax[0, 0].set_ylabel(r"$m_{\infty}$")
    ax[0, 1].axis("off")
    ax[1, 0].plot(v, h_inf(v), color='k', lw=2)
    ax[1, 0].set_ylabel(r"$h_{\infty}$")
    ax[1, 1].plot(v, tau_h(v), color='k', lw=2)
    ax[1, 1].set_ylabel(r"$\tau_{h}$ [ms]")
    ax[1, 1].set_ylim(0, 10)
    ax[2, 0].plot(v, n_inf(v), color='k', lw=2)
    ax[2, 0].set_ylabel(r"$n_{\infty}$")
    ax[2, 1].plot(v, tau_n(v), color='k', lw=2)
    ax[2, 1].set_ylabel(r"$\tau_{n}$ [ms]")
    ax[2, 1].set_ylim(0, 4)
    for i in range(3):
        for j in range(2):
            ax[i, j].tick_params(labelsize=10)
    for i in range(3):
        ax[i, 0].set_ylim(0, 1)
        ax[i, 0].set_yticks([0, 0.5, 1])
    ax[0, 0].set_xlim(np.min(v), np.max(v))
    for i in range(2):
        ax[2, i].set_xlabel("v [mV]")
        ax[2, i].set_xticks([-100, -50, 0, 50])
    plt.tight_layout()
    return fig


def plot_a_current_gates():
    v = np.arange(-100, 50, 0.01)
    fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(6, 3), sharex=True)
    ax[0, 0].plot(v, a_inf(v), color='k', lw=2)
    ax[0, 0].set_ylabel(r"$a_{\infty}(v)$")
    ax[0, 1].plot(v, np.full_like(v, tau_a(v)), color='k', lw=2)
    ax[0, 1].set_ylabel(r"$\tau_{a}$ [ms]")
    ax[1, 0].plot(v, b_inf(v), color='k', lw=2)
    ax[1, 0].set_ylabel(r"$b_{\infty}(v)$")
    ax[1, 1].plot(v, tau_b(v), color='k', lw=2)
    ax[1, 1].set_ylabel(r"$\tau_{b}$ [ms]")
    for i in range(2):
        for j in range(2):
            ax[i, j].tick_params(labelsize=10)
    for i in range(2):
        ax[i, 0].set_ylim(0, 1)
        ax[i, 0].set_yticks([0, 0.5, 1])
    ax[0, 0].set_xlim(np.min(v), np.max(v))
    for i in range(2):
        ax[1, i].set_xlabel("v [mV]")
        ax[1, i].set_xticks([-100, -50, 0, 50])
    plt.tight_layout()
    return fig


# ---------------------------------------------------------------- single-cell plots

def plot_pre_olm_voltage_trace(t, v):
    fig = plt.figure(figsize=(7, 3))
    plt.plot(t, v, lw=2, c="k")
    plt.xlim(min(t), max(t))
    plt.xlabel("time [ms]")
    plt.ylabel("v [mV]")
    plt.yticks(range(-100, 100, 50))
    plt.tight_layout()
    return fig


def plot_olm_h_current(t, v, r):
    fig, ax = plt.subplots(2, figsize=(7, 4), sharex=True)
    ax[0].plot(t, v, lw=2, c="k")
    ax[0].set_xlim(min(t), max(t))
    ax[0].set_ylabel("v [mV]")
    ax[0].set_yticks(range(-100, 100, 50))
    ax[1].plot(t, r, lw=2, c="k")
    ax[1].set_xlabel("time [ms]")
    ax[1].set_ylabel("r [mV]")
    for i in range(2):
        ax[i].tick_params(labelsize=12)
    plt.tight_layout()
    return fig


def plot_olm_h_and_a_currents(t, v, a, b):
    fig, ax = plt.subplots(2, figsize=(7, 4), sharex=True)
    ax[0].plot(t, v, lw=2, c="k")
    ax[0].set_xlim(min(t), max(t))
    ax[0].set_ylabel("v [mV]")
    ax[0].set_yticks(range(-100, 100, 50))
    ax[1].plot(t, a * b, lw=2, c="k")
    ax[1].set_xlabel("time [ms]", fontsize=14)
    ax[1].set_ylabel("ab", fontsize=14)
    for i in range(2):
        ax[i].tick_params(labelsize=12)
    plt.tight_layout()
    return fig


# ---------------------------------------------------------------- E (RTM) / I (WB) gating

def m_e_inf(v):
    alpha_m = 0.32 * (v + 54.0) / (1.0 - exp(-(v + 54.0) / 4.0))
    beta_m = 0.28 * (v + 27.0) / (exp((v + 27.0) / 5.0) - 1.0)
    return alpha_m / (alpha_m + beta_m)


def h_e_inf(v):
    alpha_h = 0.128 * exp(-(v + 50.0) / 18.0)
    beta_h = 4.0 / (1.0 + exp(-(v + 27.0) / 5.0))
    return alpha_h / (alpha_h + beta_h)


def tau_h_e(v):
    alpha_h = 0.128 * exp(-(v + 50.0) / 18.0)
    beta_h = 4.0 / (1.0 + exp(-(v + 27.0) / 5.0))
    return 1.0 / (alpha_h + beta_h)


def n_e_inf(v):
    alpha_n = 0.032 * (v + 52.0) / (1.0 - exp(-(v + 52.0) / 5.0))
    beta_n = 0.5 * exp(-(v + 57.0) / 40.0)
    return alpha_n / (alpha_n + beta_n)


def tau_n_e(v):
    alpha_n = 0.032 * (v + 52.0) / (1.0 - exp(-(v + 52.0) / 5.0))
    beta_n = 0.5 * exp(-(v + 57.0) / 40.0)
    return 1.0 / (alpha_n + beta_n)


def m_i_inf(v):
    alpha_m = 0.1 * (v + 35.0) / (1.0 - exp(-(v + 35.0) / 10.0))
    beta_m = 4.0 * exp(-(v + 60.0) / 18.0)
    return alpha_m / (alpha_m + beta_m)


def h_i_inf(v):
    alpha_h = 0.07 * exp(-(v + 58.0) / 20.0)
    beta_h = 1.0 / (exp(-0.1 * (v + 28.0)) + 1.0)
    return alpha_h / (alpha_h + beta_h)


def tau_h_i(v):
    alpha_h = 0.07 * exp(-(v + 58.0) / 20.0)
    beta_h = 1.0 / (exp(-0.1 * (v + 28.0)) + 1.0)
    return 1.0 / (alpha_h + beta_h) / 5.0


def n_i_inf(v):
    alpha_n = -0.01 * (v + 34.0) / (exp(-0.1 * (v + 34.0)) - 1.0)
    beta_n = 0.125 * exp(-(v + 44.0) / 80.0)
    return alpha_n / (alpha_n + beta_n)


def tau_n_i(v):
    alpha_n = -0.01 * (v + 34.0) / (exp(-0.1 * (v + 34.0)) - 1.0)
    beta_n = 0.125 * exp(-(v + 44.0) / 80.0)
    return 1.0 / (alpha_n + beta_n) / 5.0


# --------------------------------------------------------- double-exponential synapse

def tau_peak_function(tau_d, tau_r, tau_d_q):
    dt_ = 0.01
    dt05_ = dt_ / 2
    s, t = 0.0, 0.0
    s_inc = exp(-t / tau_d_q) * (1.0 - s) / tau_r - s * tau_d
    while s_inc > 0:
        t_old, s_inc_old = t, s_inc
        s_tmp = s + dt05_ * s_inc
        s_inc_tmp = exp(-(t + dt05_) / tau_d_q) * (1.0 - s_tmp) / tau_r - s_tmp / tau_d
        s = s + dt_ * s_inc_tmp
        t = t + dt_
        s_inc = exp(-t / tau_d_q) * (1.0 - s) / tau_r - s / tau_d
    return (t_old * (-s_inc) + t * s_inc_old) / (s_inc_old - s_inc)


def tau_d_q_function(tau_d, tau_r, tau_hat):
    tau_d_q_left = 1.0
    while tau_peak_function(tau_d, tau_r, tau_d_q_left) > tau_hat:
        tau_d_q_left /= 2.0
    tau_d_q_right = tau_r
    while tau_peak_function(tau_d, tau_r, tau_d_q_right) < tau_hat:
        tau_d_q_right *= 2.0
    while tau_d_q_right - tau_d_q_left > 1e-12:
        tau_d_q_mid = (tau_d_q_left + tau_d_q_right) / 2.0
        if tau_peak_function(tau_d, tau_r, tau_d_q_mid) <= tau_hat:
            tau_d_q_left = tau_d_q_mid
        else:
            tau_d_q_right = tau_d_q_mid
    return (tau_d_q_left + tau_d_q_right) / 2.0


# ------------------------------------------------------- population splay-state init

def rtm_init_population(i_ext, phi_vec):
    '''vectorized rtmInit over a population of RTM (E) cells: each is
    integrated (Heun) independently until its 3rd spike, then (v,h,n) is
    interpolated at phase phi_vec[i] between the 2nd and 3rd spikes.'''
    num = len(i_ext)
    max_spikes = 3
    t_final_init = 2000.0
    dt_ = 0.01
    dt05_ = dt_ / 2

    v = -70.0 * np.ones(num)
    h = h_e_inf(v)
    n = n_e_inf(v)
    t = 0.0

    num_spikes = np.zeros(num, dtype=int)
    done = np.zeros(num, dtype=bool)
    t_spikes = np.zeros((num, max_spikes))
    out = np.zeros((num, 3))

    c, g_k, g_na, g_l = 1.0, 80.0, 100.0, 0.1
    v_k, v_na, v_l = -100.0, 50.0, -67.0

    while np.sum(done) < num and t < t_final_init:
        v_old, h_old, n_old, t_old = v, h, n, t
        m = m_e_inf(v)

        v_inc = (i_ext - g_l * (v - v_l) - g_k * n ** 4 * (v - v_k)
                  - g_na * m ** 3 * h * (v - v_na)) / c
        h_inc = (h_e_inf(v) - h) / tau_h_e(v)
        n_inc = (n_e_inf(v) - n) / tau_n_e(v)

        v_tmp = v + dt05_ * v_inc
        h_tmp = h + dt05_ * h_inc
        n_tmp = n + dt05_ * n_inc
        m_tmp = m_e_inf(v_tmp)

        v_inc = (i_ext - g_l * (v_tmp - v_l) - g_k * n_tmp ** 4 * (v_tmp - v_k)
                  - g_na * m_tmp ** 3 * h_tmp * (v_tmp - v_na)) / c
        h_inc = (h_e_inf(v_tmp) - h_tmp) / tau_h_e(v_tmp)
        n_inc = (n_e_inf(v_tmp) - n_tmp) / tau_n_e(v_tmp)

        v = v + dt_ * v_inc
        h = h + dt_ * h_inc
        n = n + dt_ * n_inc
        t = t + dt_

        ind = np.where((v_old >= -20.0) & (v < -20.0))[0]
        for k in ind:
            num_spikes[k] += 1
            ts = (t_old * (v_old[k] + 20.0) + t * (-20.0 - v[k])) / (v_old[k] - v[k])
            if num_spikes[k] < 4:
                t_spikes[k, num_spikes[k] - 1] = ts

        thr = t_spikes[:, -1] + phi_vec * (t_spikes[:, -1] - t_spikes[:, -2])
        ind = np.where((num_spikes == max_spikes) & (t > thr) & (t_old <= thr) & (~done))[0]
        for k in ind:
            out[k, 0] = (v_old[k] * (t - thr[k]) + v[k] * (thr[k] - t_old)) / dt_
            out[k, 1] = (h_old[k] * (t - thr[k]) + h[k] * (thr[k] - t_old)) / dt_
            out[k, 2] = (n_old[k] * (t - thr[k]) + n[k] * (thr[k] - t_old)) / dt_
        done[ind] = True

    ind = np.where(~done)[0]
    out[ind, 0] = v[ind]
    out[ind, 1] = h[ind]
    out[ind, 2] = n[ind]
    return out


def wb_init_population(i_ext, phi_vec):
    '''same idea as rtm_init_population, for WB (I) cells.'''
    num = len(i_ext)
    max_spikes = 3
    t_final_init = 2000.0
    dt_ = 0.01
    dt05_ = dt_ / 2

    v = -70.0 * np.ones(num)
    h = h_i_inf(v)
    n = n_i_inf(v)
    t = 0.0

    num_spikes = np.zeros(num, dtype=int)
    done = np.zeros(num, dtype=bool)
    t_spikes = np.zeros((num, max_spikes))
    out = np.zeros((num, 3))

    c, g_k, g_na, g_l = 1.0, 9.0, 35.0, 0.1
    v_k, v_na, v_l = -90.0, 55.0, -65.0

    while np.sum(done) < num and t < t_final_init:
        v_old, h_old, n_old, t_old = v, h, n, t
        m = m_i_inf(v)

        v_inc = (i_ext - g_l * (v - v_l) - g_k * n ** 4 * (v - v_k)
                  - g_na * m ** 3 * h * (v - v_na)) / c
        h_inc = (h_i_inf(v) - h) / tau_h_i(v)
        n_inc = (n_i_inf(v) - n) / tau_n_i(v)

        v_tmp = v + dt05_ * v_inc
        h_tmp = h + dt05_ * h_inc
        n_tmp = n + dt05_ * n_inc
        m_tmp = m_i_inf(v_tmp)

        v_inc = (i_ext - g_l * (v_tmp - v_l) - g_k * n_tmp ** 4 * (v_tmp - v_k)
                  - g_na * m_tmp ** 3 * h_tmp * (v_tmp - v_na)) / c
        h_inc = (h_i_inf(v_tmp) - h_tmp) / tau_h_i(v_tmp)
        n_inc = (n_i_inf(v_tmp) - n_tmp) / tau_n_i(v_tmp)

        v = v + dt_ * v_inc
        h = h + dt_ * h_inc
        n = n + dt_ * n_inc
        t = t + dt_

        ind = np.where((v_old >= -20.0) & (v < -20.0))[0]
        for k in ind:
            num_spikes[k] += 1
            ts = (t_old * (v_old[k] + 20.0) + t * (-20.0 - v[k])) / (v_old[k] - v[k])
            if num_spikes[k] < 4:
                t_spikes[k, num_spikes[k] - 1] = ts

        thr = t_spikes[:, -1] + phi_vec * (t_spikes[:, -1] - t_spikes[:, -2])
        ind = np.where((num_spikes == max_spikes) & (t > thr) & (t_old <= thr) & (~done))[0]
        for k in ind:
            out[k, 0] = (v_old[k] * (t - thr[k]) + v[k] * (thr[k] - t_old)) / dt_
            out[k, 1] = (h_old[k] * (t - thr[k]) + h[k] * (thr[k] - t_old)) / dt_
            out[k, 2] = (n_old[k] * (t - thr[k]) + n[k] * (thr[k] - t_old)) / dt_
        done[ind] = True

    ind = np.where(~done)[0]
    out[ind, 0] = v[ind]
    out[ind, 1] = h[ind]
    out[ind, 2] = n[ind]
    return out


def olm_init_population(i_ext, phi_vec):
    '''same idea as rtm_init_population/wb_init_population, for the O-LM
    cell (with h- and A-currents): 6-state splay initializer (v,h,n,r,a,b).'''
    num = len(i_ext)
    max_spikes = 3
    t_final_init = 2000.0
    dt_ = 0.01
    dt05_ = dt_ / 2

    v = -70.0 * np.ones(num)
    h = h_inf(v)
    n = n_inf(v)
    r = r_inf(v)
    a = a_inf(v)
    b = b_inf(v)
    t = 0.0

    num_spikes = np.zeros(num, dtype=int)
    done = np.zeros(num, dtype=bool)
    t_spikes = np.zeros((num, max_spikes))
    out = np.zeros((num, 6))

    c = 1.3
    g_k, g_na, g_l = 23.0, 30.0, 0.05
    v_k, v_na, v_l = -100.0, 90.0, -70.0
    g_h, g_A = 12.0, 22.0
    v_h, v_A = -32.9, -90.0

    while np.sum(done) < num and t < t_final_init:
        v_old, h_old, n_old, r_old, a_old, b_old, t_old = v, h, n, r, a, b, t
        m = m_inf(v)

        v_inc = (i_ext - g_l * (v - v_l) - g_k * n ** 4 * (v - v_k)
                  - g_na * m ** 3 * h * (v - v_na) - g_h * r * (v - v_h)
                  - g_A * a * b * (v - v_A)) / c
        h_inc = (h_inf(v) - h) / tau_h(v)
        n_inc = (n_inf(v) - n) / tau_n(v)
        r_inc = (r_inf(v) - r) / tau_r(v)
        a_inc = (a_inf(v) - a) / tau_a(v)
        b_inc = (b_inf(v) - b) / tau_b(v)

        v_tmp = v + dt05_ * v_inc
        h_tmp = h + dt05_ * h_inc
        n_tmp = n + dt05_ * n_inc
        r_tmp = r + dt05_ * r_inc
        a_tmp = a + dt05_ * a_inc
        b_tmp = b + dt05_ * b_inc
        m_tmp = m_inf(v_tmp)

        v_inc = (i_ext - g_l * (v_tmp - v_l) - g_k * n_tmp ** 4 * (v_tmp - v_k)
                  - g_na * m_tmp ** 3 * h_tmp * (v_tmp - v_na) - g_h * r_tmp * (v_tmp - v_h)
                  - g_A * a_tmp * b_tmp * (v_tmp - v_A)) / c
        h_inc = (h_inf(v_tmp) - h_tmp) / tau_h(v_tmp)
        n_inc = (n_inf(v_tmp) - n_tmp) / tau_n(v_tmp)
        r_inc = (r_inf(v_tmp) - r_tmp) / tau_r(v_tmp)
        a_inc = (a_inf(v_tmp) - a_tmp) / tau_a(v_tmp)
        b_inc = (b_inf(v_tmp) - b_tmp) / tau_b(v_tmp)

        v = v + dt_ * v_inc
        h = h + dt_ * h_inc
        n = n + dt_ * n_inc
        r = r + dt_ * r_inc
        a = a + dt_ * a_inc
        b = b + dt_ * b_inc
        t = t + dt_

        ind = np.where((v_old >= -20.0) & (v < -20.0))[0]
        for k in ind:
            num_spikes[k] += 1
            ts = (t_old * (v_old[k] + 20.0) + t * (-20.0 - v[k])) / (v_old[k] - v[k])
            if num_spikes[k] < 4:
                t_spikes[k, num_spikes[k] - 1] = ts

        thr = t_spikes[:, -1] + phi_vec * (t_spikes[:, -1] - t_spikes[:, -2])
        ind = np.where((num_spikes == max_spikes) & (t > thr) & (t_old <= thr) & (~done))[0]
        for k in ind:
            out[k, 0] = (v_old[k] * (t - thr[k]) + v[k] * (thr[k] - t_old)) / dt_
            out[k, 1] = (h_old[k] * (t - thr[k]) + h[k] * (thr[k] - t_old)) / dt_
            out[k, 2] = (n_old[k] * (t - thr[k]) + n[k] * (thr[k] - t_old)) / dt_
            out[k, 3] = (r_old[k] * (t - thr[k]) + r[k] * (thr[k] - t_old)) / dt_
            out[k, 4] = (a_old[k] * (t - thr[k]) + a[k] * (thr[k] - t_old)) / dt_
            out[k, 5] = (b_old[k] * (t - thr[k]) + b[k] * (thr[k] - t_old)) / dt_
        done[ind] = True

    ind = np.where(~done)[0]
    out[ind, 0] = v[ind]
    out[ind, 1] = h[ind]
    out[ind, 2] = n[ind]
    out[ind, 3] = r[ind]
    out[ind, 4] = a[ind]
    out[ind, 5] = b[ind]
    return out


def spike_detection(t, v, threshold, dt):
    v = np.asarray(v)
    below = v[:-1] <= threshold
    above = v[1:] > threshold
    idx = np.where(below & above)[0]
    if len(idx) == 0:
        return np.empty(0)
    v0, v1 = v[idx], v[idx + 1]
    ts = (idx * dt * (v0 - threshold) + (idx + 1) * dt * (threshold - v1)) / (v0 - v1)
    return ts


# ------------------------------------------------------- E-I-OLM network (EIO_1)
#
# Note: this network's E-cell equation uses the same (unusual) sign
# convention as the original main.py/lib.py for its synaptic terms
# ("+ g @ s * (v_e - v_rev)" rather than "(v_rev - v_e)"), which makes an
# explicit fixed-step integrator diverge (verified at reduced scale); the
# implicit/adaptive `odeint` solver used by the original script handles it
# fine, so it is kept here rather than a numba fixed-step port.

def eio_derivative(x0, t, num_e, num_i, num_o, i_ext_e, i_ext_i, i_ext_o,
                    g_ee, g_ei, g_eo, g_ie, g_ii, g_io, g_oe, g_oi, g_oo,
                    v_rev_e, v_rev_i, v_rev_o,
                    tau_r_e, tau_d_e, tau_dq_e,
                    tau_r_i, tau_d_i, tau_dq_i,
                    tau_r_o, tau_d_o, tau_dq_o,
                    g_h, g_A):
    v_e, h_e = x0[:num_e], x0[num_e:2 * num_e]
    n_e = x0[2 * num_e:3 * num_e]
    q_e = x0[3 * num_e:4 * num_e]
    s_e = x0[4 * num_e:5 * num_e]

    n0 = 5 * num_e
    v_i = x0[n0:n0 + num_i]
    h_i = x0[n0 + num_i:n0 + 2 * num_i]
    n_i = x0[n0 + 2 * num_i:n0 + 3 * num_i]
    q_i = x0[n0 + 3 * num_i:n0 + 4 * num_i]
    s_i = x0[n0 + 4 * num_i:n0 + 5 * num_i]

    n1 = n0 + 5 * num_i
    v_o = x0[n1:n1 + num_o]
    h_o = x0[n1 + num_o:n1 + 2 * num_o]
    n_o = x0[n1 + 2 * num_o:n1 + 3 * num_o]
    r_o = x0[n1 + 3 * num_o:n1 + 4 * num_o]
    a_o = x0[n1 + 4 * num_o:n1 + 5 * num_o]
    b_o = x0[n1 + 5 * num_o:n1 + 6 * num_o]
    q_o = x0[n1 + 6 * num_o:n1 + 7 * num_o]
    s_o = x0[n1 + 7 * num_o:n1 + 8 * num_o]

    I_K_e = 80.0 * n_e ** 4 * (v_e + 100.0)
    I_Na_e = 100.0 * h_e * m_e_inf(v_e) ** 3 * (v_e - 50.0)
    I_L_e = 0.1 * (v_e + 67.0)
    dv_e = (i_ext_e - I_L_e - I_K_e - I_Na_e
            + (g_ee.T @ s_e) * (v_e - v_rev_e)
            + (g_ie.T @ s_i) * (v_e - v_rev_i)
            + (g_oe.T @ s_o) * (v_e - v_rev_o))
    dh_e = (h_e_inf(v_e) - h_e) / tau_h_e(v_e)
    dn_e = (n_e_inf(v_e) - n_e) / tau_n_e(v_e)
    dq_e = 0.5 * (1.0 + np.tanh(0.1 * v_e)) * 10.0 * (1.0 - q_e) - q_e / tau_dq_e
    ds_e = q_e * (1.0 - s_e) / tau_r_e - s_e / tau_d_e

    I_K_i = 9.0 * n_i ** 4 * (v_i + 90.0)
    I_Na_i = 35.0 * m_i_inf(v_i) ** 3 * h_i * (v_i - 55.0)
    I_L_i = 0.1 * (v_i + 65.0)
    dv_i = (i_ext_i - I_L_i - I_K_i - I_Na_i
            + (g_ei.T @ s_e) * (v_rev_e - v_i)
            + (g_ii.T @ s_i) * (v_rev_i - v_i)
            + (g_oi.T @ s_o) * (v_rev_o - v_i))
    dh_i = (h_i_inf(v_i) - h_i) / tau_h_i(v_i)
    dn_i = (n_i_inf(v_i) - n_i) / tau_n_i(v_i)
    dq_i = 0.5 * (1.0 + np.tanh(0.1 * v_i)) * 10.0 * (1.0 - q_i) - q_i / tau_dq_i
    ds_i = q_i * (1.0 - s_i) / tau_r_i - s_i / tau_d_i

    I_K_o = 23.0 * n_o ** 4 * (v_o + 100.0)
    I_Na_o = 30.0 * h_o * m_inf(v_o) ** 3 * (v_o - 90.0)
    I_L_o = 0.05 * (v_o + 70.0)
    I_H_o = g_h * r_o * (v_o + 32.9)
    I_A_o = g_A * a_o * b_o * (v_o + 90.0)
    dv_o = (i_ext_o - I_L_o - I_K_o - I_Na_o - I_H_o - I_A_o
            + (g_eo.T @ s_e) * (v_rev_e - v_o)
            + (g_io.T @ s_i) * (v_rev_i - v_o)
            + (g_oo.T @ s_o) * (v_rev_o - v_o)) / 1.3
    dh_o = (h_inf(v_o) - h_o) / tau_h(v_o)
    dn_o = (n_inf(v_o) - n_o) / tau_n(v_o)
    dr_o = (r_inf(v_o) - r_o) / tau_r(v_o)
    da_o = (a_inf(v_o) - a_o) / tau_a(v_o)
    db_o = (b_inf(v_o) - b_o) / tau_b(v_o)
    dq_o = 0.5 * (1.0 + np.tanh(0.1 * v_o)) * (1.0 - q_o) / 0.1 - q_o / tau_dq_o
    ds_o = q_o * (1.0 - s_o) / tau_r_o - s_o / tau_d_o

    return np.hstack((dv_e, dh_e, dn_e, dq_e, ds_e,
                       dv_i, dh_i, dn_i, dq_i, ds_i,
                       dv_o, dh_o, dn_o, dr_o, da_o, db_o, dq_o, ds_o))


def simulate_eio_network(num_e=10, num_i=5, num_o=5,
                          g_hat_ei=0.25, g_hat_ie=0.25, g_hat_ii=0.25,
                          g_hat_io=0.50, g_hat_oe=1.00, g_hat_oi=0.50,
                          g_h=12.0, g_A=22.0, t_final=1000.0, dt=0.01, seed=0):
    '''E-I-OLM population network (EIO_1): num_e RTM E-cells, num_i WB
    I-cells, and num_o OLM cells (with h- and A-currents), coupled by
    double-exponential synapses. Faithful port of main.py/lib.py (odeint
    integration, matching the original).'''
    rng = np.random.default_rng(seed)

    sigma_e, sigma_i, sigma_o = 0.05, 0.10, 0.05
    i_ext_e = 1.8 * np.ones(num_e) * (1 + sigma_e * rng.standard_normal(num_e))
    i_ext_i = 1.0 * np.ones(num_i) * (1 + sigma_i * rng.standard_normal(num_i))
    i_ext_o = -2.0 * np.ones(num_o) * (1 + sigma_o * rng.standard_normal(num_o))

    def conn(g_hat, n_pre, n_post, p=1.0):
        if g_hat == 0.0:
            return np.zeros((n_pre, n_post))
        u = rng.random((n_pre, n_post))
        return g_hat * (u < p) / (n_pre * p)

    g_ee = conn(0.0, num_e, num_e)
    g_ei = conn(g_hat_ei, num_e, num_i)
    g_eo = conn(0.0, num_e, num_o)
    g_ie = conn(g_hat_ie, num_i, num_e)
    g_ii = conn(g_hat_ii, num_i, num_i)
    g_io = conn(g_hat_io, num_i, num_o)
    g_oe = conn(g_hat_oe, num_o, num_e)
    g_oi = conn(g_hat_oi, num_o, num_i)
    g_oo = conn(0.0, num_o, num_o)

    v_rev_e, v_rev_i, v_rev_o = 0.0, -75.0, -75.0
    tau_r_e, tau_d_e = 0.5, 3.0
    tau_r_i, tau_d_i = 0.5, 9.0
    tau_r_o, tau_d_o = 0.5, 20.0
    tau_peak = 0.5
    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak)
    tau_dq_o = tau_d_q_function(tau_d_o, tau_r_o, tau_peak)

    iv = rtm_init_population(i_ext_e, rng.random(num_e))
    v_e, h_e, n_e = iv[:, 0], iv[:, 1], iv[:, 2]
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)

    iv = wb_init_population(i_ext_i, rng.random(num_i))
    v_i, h_i, n_i = iv[:, 0], iv[:, 1], iv[:, 2]
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    iv = olm_init_population(i_ext_o, rng.random(num_o))
    v_o, h_o, n_o, r_o, a_o, b_o = iv[:, 0], iv[:, 1], iv[:, 2], iv[:, 3], iv[:, 4], iv[:, 5]
    q_o, s_o = np.zeros(num_o), np.zeros(num_o)

    x0 = np.hstack((v_e, h_e, n_e, q_e, s_e,
                     v_i, h_i, n_i, q_i, s_i,
                     v_o, h_o, n_o, r_o, a_o, b_o, q_o, s_o))
    t = np.arange(0, t_final, dt)
    sol = odeint(eio_derivative, x0, t, args=(
        num_e, num_i, num_o, i_ext_e, i_ext_i, i_ext_o,
        g_ee, g_ei, g_eo, g_ie, g_ii, g_io, g_oe, g_oi, g_oo,
        v_rev_e, v_rev_i, v_rev_o,
        tau_r_e, tau_d_e, tau_dq_e,
        tau_r_i, tau_d_i, tau_dq_i,
        tau_r_o, tau_d_o, tau_dq_o,
        g_h, g_A))

    lfp_v = sol[:, :num_e].mean(axis=1)
    lfp_s = sol[:, 4 * num_e:5 * num_e].mean(axis=1)

    t_e_spikes = [spike_detection(t, sol[:, i], -20.0, dt) for i in range(num_e)]
    idx = 5 * num_e
    t_i_spikes = [spike_detection(t, sol[:, idx + i], -20.0, dt) for i in range(num_i)]
    idx = idx + 5 * num_i
    t_o_spikes = [spike_detection(t, sol[:, idx + i], -20.0, dt) for i in range(num_o)]

    return t_e_spikes, t_i_spikes, t_o_spikes, t, lfp_v, lfp_s, num_e, num_i, num_o


def plot_eio_raster(t_e_spikes, t_i_spikes, t_o_spikes, t, lfp_v, lfp_s, num_e, num_i, num_o):
    '''3-panel raster (O, I, E cells stacked) + mean(v_E) + mean(s_E), like
    the original rasterogram.py.'''
    fig, ax = plt.subplots(3, figsize=(10, 8), sharex=True)
    for i in range(num_o):
        ax[0].plot(t_o_spikes[i], [i] * len(t_o_spikes[i]), "g.")
    for i in range(num_o, num_o + num_i):
        ax[0].plot(t_i_spikes[i - num_o], [i] * len(t_i_spikes[i - num_o]), "b.")
    index = num_o + num_i
    for i in range(index, index + num_e):
        ax[0].plot(t_e_spikes[i - index], [i] * len(t_e_spikes[i - index]), "r.")
    ax[1].plot(t, lfp_v, lw=2, color="k")
    ax[2].plot(t, lfp_s, lw=2, color="k")
    ax[2].set_xlabel("time [ms]", fontsize=14)
    ax[0].set_ylabel("neuron #", fontsize=14)
    ax[1].set_ylabel(r"mean($v_E$)", fontsize=14)
    ax[2].set_ylabel(r"mean($s_E$)", fontsize=14)
    for i in range(3):
        ax[i].tick_params(labelsize=11)
    ax[0].set_xlim(0, np.max(t))
    plt.tight_layout()
    return fig


# ------------------------------------------------------- numba PING+theta network

@njit
def _m_e_inf_s(v):
    alpha_m = 0.32 * (v + 54.0) / (1.0 - math.exp(-(v + 54.0) / 4.0))
    beta_m = 0.28 * (v + 27.0) / (math.exp((v + 27.0) / 5.0) - 1.0)
    return alpha_m / (alpha_m + beta_m)


@njit
def _h_e_inf_s(v):
    alpha_h = 0.128 * math.exp(-(v + 50.0) / 18.0)
    beta_h = 4.0 / (1.0 + math.exp(-(v + 27.0) / 5.0))
    return alpha_h / (alpha_h + beta_h)


@njit
def _tau_h_e_s(v):
    alpha_h = 0.128 * math.exp(-(v + 50.0) / 18.0)
    beta_h = 4.0 / (1.0 + math.exp(-(v + 27.0) / 5.0))
    return 1.0 / (alpha_h + beta_h)


@njit
def _n_e_inf_s(v):
    alpha_n = 0.032 * (v + 52.0) / (1.0 - math.exp(-(v + 52.0) / 5.0))
    beta_n = 0.5 * math.exp(-(v + 57.0) / 40.0)
    return alpha_n / (alpha_n + beta_n)


@njit
def _tau_n_e_s(v):
    alpha_n = 0.032 * (v + 52.0) / (1.0 - math.exp(-(v + 52.0) / 5.0))
    beta_n = 0.5 * math.exp(-(v + 57.0) / 40.0)
    return 1.0 / (alpha_n + beta_n)


@njit
def _m_i_inf_s(v):
    alpha_m = 0.1 * (v + 35.0) / (1.0 - math.exp(-(v + 35.0) / 10.0))
    beta_m = 4.0 * math.exp(-(v + 60.0) / 18.0)
    return alpha_m / (alpha_m + beta_m)


@njit
def _h_i_inf_s(v):
    alpha_h = 0.07 * math.exp(-(v + 58.0) / 20.0)
    beta_h = 1.0 / (math.exp(-0.1 * (v + 28.0)) + 1.0)
    return alpha_h / (alpha_h + beta_h)


@njit
def _tau_h_i_s(v):
    alpha_h = 0.07 * math.exp(-(v + 58.0) / 20.0)
    beta_h = 1.0 / (math.exp(-0.1 * (v + 28.0)) + 1.0)
    return 1.0 / (alpha_h + beta_h) / 5.0


@njit
def _n_i_inf_s(v):
    alpha_n = -0.01 * (v + 34.0) / (math.exp(-0.1 * (v + 34.0)) - 1.0)
    beta_n = 0.125 * math.exp(-(v + 44.0) / 80.0)
    return alpha_n / (alpha_n + beta_n)


@njit
def _tau_n_i_s(v):
    alpha_n = -0.01 * (v + 34.0) / (math.exp(-0.1 * (v + 34.0)) - 1.0)
    beta_n = 0.125 * math.exp(-(v + 44.0) / 80.0)
    return 1.0 / (alpha_n + beta_n) / 5.0


@njit
def _theta_drive_step_loop(m_steps, dt, dt05, num_e, num_i,
                            v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
                            tau_r_i, tau_d_i, tau_dq_i,
                            P, alpha,
                            i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
                            v_e, h_e, n_e, q_e, s_e,
                            v_i, h_i, n_i, q_i, s_i):
    e_times = List.empty_list(np.float64)
    e_indices = List.empty_list(np.int64)
    i_times = List.empty_list(np.float64)
    i_indices = List.empty_list(np.int64)

    lfp_v = np.empty(m_steps + 1)
    lfp_s = np.empty(m_steps + 1)
    lfp_v[0] = v_e.mean()
    lfp_s[0] = s_e.mean()

    two_pi = 2.0 * math.pi

    for step in range(m_steps):
        k = step + 1
        t0 = step * dt
        drive0 = 1.0 + alpha * math.sin(two_pi * t0 / P)
        drive_mid = 1.0 + alpha * math.sin(two_pi * (t0 + dt05) / P)

        ee = g_ee.T @ s_e
        ie = g_ie.T @ s_i
        ei = g_ei.T @ s_e
        ii = g_ii.T @ s_i

        dve = np.empty(num_e); dhe = np.empty(num_e); dne = np.empty(num_e)
        dqe = np.empty(num_e); dse = np.empty(num_e)
        for j in range(num_e):
            v = v_e[j]
            m = _m_e_inf_s(v)
            I_L = 0.1 * (v + 67.0)
            I_K = 80.0 * math.pow(n_e[j], 4.0) * (v + 100.0)
            I_Na = 100.0 * math.pow(m, 3.0) * h_e[j] * (v - 50.0)
            dve[j] = (i_ext_e[j] * drive0 - I_L - I_K - I_Na
                      + ee[j] * (v_rev_e - v) + ie[j] * (v_rev_i - v))
            dhe[j] = (_h_e_inf_s(v) - h_e[j]) / _tau_h_e_s(v)
            dne[j] = (_n_e_inf_s(v) - n_e[j]) / _tau_n_e_s(v)
            th = math.tanh(0.1 * v)
            dqe[j] = 0.5 * (1.0 + th) * 10.0 * (1.0 - q_e[j]) - q_e[j] / tau_dq_e
            dse[j] = q_e[j] * (1.0 - s_e[j]) / tau_r_e - s_e[j] / tau_d_e

        dvi = np.empty(num_i); dhi = np.empty(num_i); dni = np.empty(num_i)
        dqi = np.empty(num_i); dsi = np.empty(num_i)
        for j in range(num_i):
            v = v_i[j]
            m = _m_i_inf_s(v)
            I_L = 0.1 * (v + 65.0)
            I_K = 9.0 * math.pow(n_i[j], 4.0) * (v + 90.0)
            I_Na = 35.0 * math.pow(m, 3.0) * h_i[j] * (v - 55.0)
            dvi[j] = (i_ext_i[j] - I_L - I_K - I_Na
                      + ei[j] * (v_rev_e - v) + ii[j] * (v_rev_i - v))
            dhi[j] = (_h_i_inf_s(v) - h_i[j]) / _tau_h_i_s(v)
            dni[j] = (_n_i_inf_s(v) - n_i[j]) / _tau_n_i_s(v)
            th = math.tanh(0.1 * v)
            dqi[j] = 0.5 * (1.0 + th) * 10.0 * (1.0 - q_i[j]) - q_i[j] / tau_dq_i
            dsi[j] = q_i[j] * (1.0 - s_i[j]) / tau_r_i - s_i[j] / tau_d_i

        ve_m = v_e + dt05 * dve; he_m = h_e + dt05 * dhe; ne_m = n_e + dt05 * dne
        qe_m = q_e + dt05 * dqe; se_m = s_e + dt05 * dse
        vi_m = v_i + dt05 * dvi; hi_m = h_i + dt05 * dhi; ni_m = n_i + dt05 * dni
        qi_m = q_i + dt05 * dqi; si_m = s_i + dt05 * dsi

        ee = g_ee.T @ se_m
        ie = g_ie.T @ si_m
        ei = g_ei.T @ se_m
        ii = g_ii.T @ si_m

        for j in range(num_e):
            v = ve_m[j]
            m = _m_e_inf_s(v)
            I_L = 0.1 * (v + 67.0)
            I_K = 80.0 * math.pow(ne_m[j], 4.0) * (v + 100.0)
            I_Na = 100.0 * math.pow(m, 3.0) * he_m[j] * (v - 50.0)
            dve[j] = (i_ext_e[j] * drive_mid - I_L - I_K - I_Na
                      + ee[j] * (v_rev_e - v) + ie[j] * (v_rev_i - v))
            dhe[j] = (_h_e_inf_s(v) - he_m[j]) / _tau_h_e_s(v)
            dne[j] = (_n_e_inf_s(v) - ne_m[j]) / _tau_n_e_s(v)
            th = math.tanh(0.1 * v)
            dqe[j] = 0.5 * (1.0 + th) * 10.0 * (1.0 - qe_m[j]) - qe_m[j] / tau_dq_e
            dse[j] = qe_m[j] * (1.0 - se_m[j]) / tau_r_e - se_m[j] / tau_d_e

        for j in range(num_i):
            v = vi_m[j]
            m = _m_i_inf_s(v)
            I_L = 0.1 * (v + 65.0)
            I_K = 9.0 * math.pow(ni_m[j], 4.0) * (v + 90.0)
            I_Na = 35.0 * math.pow(m, 3.0) * hi_m[j] * (v - 55.0)
            dvi[j] = (i_ext_i[j] - I_L - I_K - I_Na
                      + ei[j] * (v_rev_e - v) + ii[j] * (v_rev_i - v))
            dhi[j] = (_h_i_inf_s(v) - hi_m[j]) / _tau_h_i_s(v)
            dni[j] = (_n_i_inf_s(v) - ni_m[j]) / _tau_n_i_s(v)
            th = math.tanh(0.1 * v)
            dqi[j] = 0.5 * (1.0 + th) * 10.0 * (1.0 - qi_m[j]) - qi_m[j] / tau_dq_i
            dsi[j] = qi_m[j] * (1.0 - si_m[j]) / tau_r_i - si_m[j] / tau_d_i

        ve_old = v_e.copy(); vi_old = v_i.copy()
        v_e[:] = v_e + dt * dve; h_e[:] = h_e + dt * dhe; n_e[:] = n_e + dt * dne
        q_e[:] = q_e + dt * dqe; s_e[:] = s_e + dt * dse
        v_i[:] = v_i + dt * dvi; h_i[:] = h_i + dt * dhi; n_i[:] = n_i + dt * dni
        q_i[:] = q_i + dt * dqi; s_i[:] = s_i + dt * dsi

        for j in range(num_e):
            if ve_old[j] <= -20.0 and v_e[j] > -20.0:
                e_indices.append(j)
                e_times.append(((-20.0 - ve_old[j]) * k * dt + (v_e[j] + 20.0) * step * dt)
                                / (v_e[j] - ve_old[j]))
        for j in range(num_i):
            if vi_old[j] <= -20.0 and v_i[j] > -20.0:
                i_indices.append(j)
                i_times.append(((-20.0 - vi_old[j]) * k * dt + (v_i[j] + 20.0) * step * dt)
                                / (v_i[j] - vi_old[j]))

        lfp_v[k] = v_e.mean()
        lfp_s[k] = s_e.mean()

    return e_times, e_indices, i_times, i_indices, lfp_v, lfp_s


def simulate_ping_theta_drive(num_e=40, num_i=10, alpha=0.8, P=125.0,
                               g_hat_ei=0.25, g_hat_ie=0.25, g_hat_ii=0.25,
                               t_final=1000.0, dt=0.01, seed=0):
    '''PING network with a theta-periodic E-cell drive: i_ext_e is
    multiplied by (1 + alpha*sin(2*pi*t/P)), opening a gamma-permissive
    window once per theta cycle. Faithful port of main.py/lib.py, with the
    network stepper (fixed-step Heun, dt matching the original) numba
    -accelerated.'''
    rng = np.random.default_rng(seed)
    sigma_e = 0.05
    i_ext_e = 1.4 * np.ones(num_e) * (1 + sigma_e * rng.standard_normal(num_e))
    i_ext_i = np.zeros(num_i)

    def conn(g_hat, n_pre, n_post):
        if g_hat == 0.0:
            return np.zeros((n_pre, n_post))
        u = rng.random((n_pre, n_post))
        return g_hat * (u < 1.0) / n_pre

    g_ee = conn(0.0, num_e, num_e)
    g_ei = conn(g_hat_ei, num_e, num_i)
    g_ie = conn(g_hat_ie, num_i, num_e)
    g_ii = conn(g_hat_ii, num_i, num_i)

    v_rev_e, v_rev_i = 0.0, -75.0
    tau_r_e, tau_d_e = 0.5, 3.0
    tau_r_i, tau_d_i = 0.5, 9.0
    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, 0.5)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, 0.5)

    iv = rtm_init_population(i_ext_e, rng.random(num_e))
    v_e, h_e, n_e = iv[:, 0].copy(), iv[:, 1].copy(), iv[:, 2].copy()
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)

    v_i = -75.0 * np.ones(num_i)
    h_i, n_i = h_i_inf(v_i), n_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    dt05 = dt / 2
    m_steps = round(t_final / dt)
    (t_e, i_e, t_i, i_i, lfp_v, lfp_s) = _theta_drive_step_loop(
        m_steps, dt, dt05, num_e, num_i,
        v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
        tau_r_i, tau_d_i, tau_dq_i, P, alpha,
        i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
        v_e, h_e, n_e, q_e, s_e, v_i, h_i, n_i, q_i, s_i)

    t_e = np.array(t_e) if len(t_e) else np.empty(0)
    i_e = np.array(i_e, dtype=int) if len(i_e) else np.empty(0, dtype=int)
    t_i = np.array(t_i) if len(t_i) else np.empty(0)
    i_i = np.array(i_i, dtype=int) if len(i_i) else np.empty(0, dtype=int)
    t_axis = np.arange(m_steps + 1) * dt
    return t_e, i_e, t_i, i_i, t_axis, np.array(lfp_v), np.array(lfp_s), num_e, num_i


def plot_ping_theta_raster(t_e, i_e, t_i, i_i, t, lfp_v, lfp_s, num_e, num_i):
    fig, ax = plt.subplots(3, figsize=(10, 8), sharex=True)
    if len(t_i):
        ax[0].plot(t_i, i_i, 'b.')
    if len(t_e):
        ax[0].plot(t_e, i_e + num_i, 'r.')
    ax[1].plot(t, lfp_v, lw=2, color="k")
    ax[2].plot(t, lfp_s, lw=2, color="k")
    ax[2].set_xlabel("time [ms]", fontsize=14)
    ax[0].set_ylabel("neuron #", fontsize=14)
    ax[1].set_ylabel(r"mean($v_E$)", fontsize=14)
    ax[2].set_ylabel(r"mean($s_E$)", fontsize=14)
    for i in range(3):
        ax[i].tick_params(labelsize=11)
    ax[0].set_xlim(0, np.max(t))
    plt.tight_layout()
    return fig


@njit
def _theta_inhibition_step_loop(m_steps, dt, dt05, num_e, num_i,
                                 v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
                                 tau_r_i, tau_d_i, tau_dq_i,
                                 P, g_ex,
                                 i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
                                 v_e, h_e, n_e, q_e, s_e,
                                 v_i, h_i, n_i, q_i, s_i):
    e_times = List.empty_list(np.float64)
    e_indices = List.empty_list(np.int64)
    i_times = List.empty_list(np.float64)
    i_indices = List.empty_list(np.int64)

    lfp_v = np.empty(m_steps + 1)
    lfp_s = np.empty(m_steps + 1)
    lfp_v[0] = v_e.mean()
    lfp_s[0] = s_e.mean()

    for step in range(m_steps):
        k = step + 1
        t0 = step * dt
        s0 = math.sin(math.pi * t0 / P)
        alpha_ex0 = math.exp(-10.0 * s0 * s0)
        s_mid = math.sin(math.pi * (t0 + dt05) / P)
        alpha_ex_mid = math.exp(-10.0 * s_mid * s_mid)

        ee = g_ee.T @ s_e
        ie = g_ie.T @ s_i
        ei = g_ei.T @ s_e
        ii = g_ii.T @ s_i

        dve = np.empty(num_e); dhe = np.empty(num_e); dne = np.empty(num_e)
        dqe = np.empty(num_e); dse = np.empty(num_e)
        for j in range(num_e):
            v = v_e[j]
            m = _m_e_inf_s(v)
            I_L = 0.1 * (v + 67.0)
            I_K = 80.0 * math.pow(n_e[j], 4.0) * (v + 100.0)
            I_Na = 100.0 * math.pow(m, 3.0) * h_e[j] * (v - 50.0)
            dve[j] = (i_ext_e[j] - I_L - I_K - I_Na
                      + ee[j] * (v_rev_e - v) + ie[j] * (v_rev_i - v)
                      + g_ex * alpha_ex0 * (v_rev_i - v))
            dhe[j] = (_h_e_inf_s(v) - h_e[j]) / _tau_h_e_s(v)
            dne[j] = (_n_e_inf_s(v) - n_e[j]) / _tau_n_e_s(v)
            th = math.tanh(0.1 * v)
            dqe[j] = 0.5 * (1.0 + th) * 10.0 * (1.0 - q_e[j]) - q_e[j] / tau_dq_e
            dse[j] = q_e[j] * (1.0 - s_e[j]) / tau_r_e - s_e[j] / tau_d_e

        dvi = np.empty(num_i); dhi = np.empty(num_i); dni = np.empty(num_i)
        dqi = np.empty(num_i); dsi = np.empty(num_i)
        for j in range(num_i):
            v = v_i[j]
            m = _m_i_inf_s(v)
            I_L = 0.1 * (v + 65.0)
            I_K = 9.0 * math.pow(n_i[j], 4.0) * (v + 90.0)
            I_Na = 35.0 * math.pow(m, 3.0) * h_i[j] * (v - 55.0)
            dvi[j] = (i_ext_i[j] - I_L - I_K - I_Na
                      + ei[j] * (v_rev_e - v) + ii[j] * (v_rev_i - v))
            dhi[j] = (_h_i_inf_s(v) - h_i[j]) / _tau_h_i_s(v)
            dni[j] = (_n_i_inf_s(v) - n_i[j]) / _tau_n_i_s(v)
            th = math.tanh(0.1 * v)
            dqi[j] = 0.5 * (1.0 + th) * 10.0 * (1.0 - q_i[j]) - q_i[j] / tau_dq_i
            dsi[j] = q_i[j] * (1.0 - s_i[j]) / tau_r_i - s_i[j] / tau_d_i

        ve_m = v_e + dt05 * dve; he_m = h_e + dt05 * dhe; ne_m = n_e + dt05 * dne
        qe_m = q_e + dt05 * dqe; se_m = s_e + dt05 * dse
        vi_m = v_i + dt05 * dvi; hi_m = h_i + dt05 * dhi; ni_m = n_i + dt05 * dni
        qi_m = q_i + dt05 * dqi; si_m = s_i + dt05 * dsi

        ee = g_ee.T @ se_m
        ie = g_ie.T @ si_m
        ei = g_ei.T @ se_m
        ii = g_ii.T @ si_m

        for j in range(num_e):
            v = ve_m[j]
            m = _m_e_inf_s(v)
            I_L = 0.1 * (v + 67.0)
            I_K = 80.0 * math.pow(ne_m[j], 4.0) * (v + 100.0)
            I_Na = 100.0 * math.pow(m, 3.0) * he_m[j] * (v - 50.0)
            dve[j] = (i_ext_e[j] - I_L - I_K - I_Na
                      + ee[j] * (v_rev_e - v) + ie[j] * (v_rev_i - v)
                      + g_ex * alpha_ex_mid * (v_rev_i - v))
            dhe[j] = (_h_e_inf_s(v) - he_m[j]) / _tau_h_e_s(v)
            dne[j] = (_n_e_inf_s(v) - ne_m[j]) / _tau_n_e_s(v)
            th = math.tanh(0.1 * v)
            dqe[j] = 0.5 * (1.0 + th) * 10.0 * (1.0 - qe_m[j]) - qe_m[j] / tau_dq_e
            dse[j] = qe_m[j] * (1.0 - se_m[j]) / tau_r_e - se_m[j] / tau_d_e

        for j in range(num_i):
            v = vi_m[j]
            m = _m_i_inf_s(v)
            I_L = 0.1 * (v + 65.0)
            I_K = 9.0 * math.pow(ni_m[j], 4.0) * (v + 90.0)
            I_Na = 35.0 * math.pow(m, 3.0) * hi_m[j] * (v - 55.0)
            dvi[j] = (i_ext_i[j] - I_L - I_K - I_Na
                      + ei[j] * (v_rev_e - v) + ii[j] * (v_rev_i - v))
            dhi[j] = (_h_i_inf_s(v) - hi_m[j]) / _tau_h_i_s(v)
            dni[j] = (_n_i_inf_s(v) - ni_m[j]) / _tau_n_i_s(v)
            th = math.tanh(0.1 * v)
            dqi[j] = 0.5 * (1.0 + th) * 10.0 * (1.0 - qi_m[j]) - qi_m[j] / tau_dq_i
            dsi[j] = qi_m[j] * (1.0 - si_m[j]) / tau_r_i - si_m[j] / tau_d_i

        ve_old = v_e.copy(); vi_old = v_i.copy()
        v_e[:] = v_e + dt * dve; h_e[:] = h_e + dt * dhe; n_e[:] = n_e + dt * dne
        q_e[:] = q_e + dt * dqe; s_e[:] = s_e + dt * dse
        v_i[:] = v_i + dt * dvi; h_i[:] = h_i + dt * dhi; n_i[:] = n_i + dt * dni
        q_i[:] = q_i + dt * dqi; s_i[:] = s_i + dt * dsi

        for j in range(num_e):
            if ve_old[j] <= -20.0 and v_e[j] > -20.0:
                e_indices.append(j)
                e_times.append(((-20.0 - ve_old[j]) * k * dt + (v_e[j] + 20.0) * step * dt)
                                / (v_e[j] - ve_old[j]))
        for j in range(num_i):
            if vi_old[j] <= -20.0 and v_i[j] > -20.0:
                i_indices.append(j)
                i_times.append(((-20.0 - vi_old[j]) * k * dt + (v_i[j] + 20.0) * step * dt)
                                / (v_i[j] - vi_old[j]))

        lfp_v[k] = v_e.mean()
        lfp_s[k] = s_e.mean()

    return e_times, e_indices, i_times, i_indices, lfp_v, lfp_s


def simulate_ping_theta_inhibition(num_e=40, num_i=10, g_ex=0.2, P=125.0,
                                    g_hat_ei=0.25, g_hat_ie=0.25, g_hat_ii=0.25,
                                    t_final=1000.0, dt=0.01, seed=0):
    '''PING network with a theta-periodic external inhibitory conductance
    onto the E cells (g_ex * alpha_ex(t) * (v_rev_i - v_e), a narrow
    inhibitory pulse once per theta cycle), rather than a modulated drive.
    Faithful port of main.py/lib.py, numba-accelerated network stepper.'''
    rng = np.random.default_rng(seed)
    sigma_e = 0.05
    i_ext_e = 1.4 * np.ones(num_e) * (1 + sigma_e * rng.standard_normal(num_e))
    i_ext_i = np.zeros(num_i)

    def conn(g_hat, n_pre, n_post):
        if g_hat == 0.0:
            return np.zeros((n_pre, n_post))
        u = rng.random((n_pre, n_post))
        return g_hat * (u < 1.0) / n_pre

    g_ee = conn(0.0, num_e, num_e)
    g_ei = conn(g_hat_ei, num_e, num_i)
    g_ie = conn(g_hat_ie, num_i, num_e)
    g_ii = conn(g_hat_ii, num_i, num_i)

    v_rev_e, v_rev_i = 0.0, -75.0
    tau_r_e, tau_d_e = 0.5, 3.0
    tau_r_i, tau_d_i = 0.5, 9.0
    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, 0.5)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, 0.5)

    iv = rtm_init_population(i_ext_e, rng.random(num_e))
    v_e, h_e, n_e = iv[:, 0].copy(), iv[:, 1].copy(), iv[:, 2].copy()
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)

    v_i = -75.0 * np.ones(num_i)
    h_i, n_i = h_i_inf(v_i), n_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    dt05 = dt / 2
    m_steps = round(t_final / dt)
    (t_e, i_e, t_i, i_i, lfp_v, lfp_s) = _theta_inhibition_step_loop(
        m_steps, dt, dt05, num_e, num_i,
        v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
        tau_r_i, tau_d_i, tau_dq_i, P, g_ex,
        i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
        v_e, h_e, n_e, q_e, s_e, v_i, h_i, n_i, q_i, s_i)

    t_e = np.array(t_e) if len(t_e) else np.empty(0)
    i_e = np.array(i_e, dtype=int) if len(i_e) else np.empty(0, dtype=int)
    t_i = np.array(t_i) if len(t_i) else np.empty(0)
    i_i = np.array(i_i, dtype=int) if len(i_i) else np.empty(0, dtype=int)
    t_axis = np.arange(m_steps + 1) * dt
    return t_e, i_e, t_i, i_i, t_axis, np.array(lfp_v), np.array(lfp_s), num_e, num_i


## A-current

`plot_a_current_gates` plots the O-LM cell's A-current activation/
inactivation gates, $a_\infty(v)$/$b_\infty(v)$, and their time constants
$\tau_a$ (a flat 5 ms) and $\tau_b(v)$.

In [ ]:
fig = plot_a_current_gates()
plt.show()

## E-I-O-LM Network (EIO_1)

`simulate_eio_network` builds a small E-I-O-LM population (10 RTM E-cells,
5 WB I-cells, 5 O-LM cells) coupled by double-exponential synapses, with the
O-LM population inhibiting the E and I populations and receiving excitation
back from the E population -- closing a loop that can pace gamma episodes at
a slower rhythm. `plot_eio_raster` draws the three-population raster (O-LM
in green, I in blue, E in red) together with mean($v_E$) and mean($s_E$),
matching the original `rasterogram.py`. This is the slowest simulation in
the chapter (an `odeint`-integrated 115-dimensional system) and can take a
few minutes.

In [ ]:
res = simulate_eio_network()
fig = plot_eio_raster(*res)
plt.show()

The O-cell $\to$ E-cell coupling strength `g_hat_oe` sets how strongly the
O-LM population's theta-paced inhibition reaches the E cells; the slider
below reruns a shorter (300 ms) version of the network to stay responsive.

In [ ]:
interact(lambda g_hat_oe=1.0: plot_eio_raster(*simulate_eio_network(g_hat_oe=g_hat_oe, t_final=300.0)),
          g_hat_oe=(0.0, 2.0, 0.1));

## O-LM Cell with h- and A-currents

`simulate_olm_h_and_a_currents` integrates a single O-LM cell with both the
h-current and the A-current active, and `plot_olm_h_and_a_currents` plots
$v(t)$ together with the A-current's combined gate $a(t)b(t)$. The A-current
strength `g_A` sets how strongly the transient A-current delays/shapes the
cell's response.

In [ ]:
interact(lambda g_A=22.0: plot_olm_h_and_a_currents(*simulate_olm_h_and_a_currents(g_A=g_A)),
          g_A=(0.0, 40.0, 2.0));

## O-LM Cell with h-current

`simulate_olm_h_current` integrates a single O-LM cell with only the
h-current active (no A-current), and `plot_olm_h_current` plots $v(t)$
together with the h-current gate $r(t)$. The h-current strength `g_h`
controls the slow post-inhibitory-rebound dynamics.

In [ ]:
interact(lambda g_h=12.0: plot_olm_h_current(*simulate_olm_h_current(g_h=g_h)),
          g_h=(0.0, 24.0, 1.0));

## PING with Theta-Modulated Drive

`simulate_ping_theta_drive` runs a 40 E-cell/10 I-cell PING network whose
E-cell drive `i_ext_e` is multiplied by $(1+\alpha\sin(2\pi t/P))$ --
a theta-periodic (`P`, default 125 ms) boost that opens a gamma-permissive
window once per theta cycle. `plot_ping_theta_raster` shows the raster (I
cells below, E cells above) with mean($v_E$) and mean($s_E$) beneath it. The
modulation depth `alpha` controls how sharply gamma is confined to the peak
of each theta cycle.

In [ ]:
interact(lambda alpha=0.8: plot_ping_theta_raster(*simulate_ping_theta_drive(alpha=alpha)),
          alpha=(0.0, 1.0, 0.05));

## PING with Theta-Modulated Inhibition

`simulate_ping_theta_inhibition` runs the same 40/10-cell PING network, but
instead of modulating the E-cell drive, it adds an external inhibitory
conductance $g_{ex}\,\alpha_{ex}(t)\,(E_I - v_E)$ onto the E cells, where
$\alpha_{ex}(t)=\exp(-10\sin^2(\pi t/P))$ is a narrow inhibitory pulse once
per theta cycle. `plot_ping_theta_raster` (shared with the drive-modulated
version above) plots the resulting raster and LFP-like traces. The
inhibition strength `g_ex` sets how sharply that pulse suppresses E-cell
firing outside its gamma window.

In [ ]:
interact(lambda g_ex=0.2: plot_ping_theta_raster(*simulate_ping_theta_inhibition(g_ex=g_ex)),
          g_ex=(0.0, 0.5, 0.02));

## Pre-O-LM Voltage Trace

`simulate_pre_olm_voltage_trace` integrates a single pre-O-LM cell (the same
fast gates as the O-LM cell, but without h- or A-currents) under a constant
current `i_ext`, and `plot_pre_olm_voltage_trace` plots $v(t)$.

In [ ]:
interact(lambda i_ext=1.5: plot_pre_olm_voltage_trace(*simulate_pre_olm_voltage_trace(i_ext=i_ext)),
          i_ext=(0.0, 3.0, 0.1));

## Pre-O-LM Gating Variables

`plot_pre_olm_gates` plots the pre-O-LM cell's steady-state gates
$m_\infty$, $h_\infty$, $n_\infty$ and time constants $\tau_h$, $\tau_n$ --
the fast-gate building block shared by every O-LM-family cell in this
chapter.

In [ ]:
fig = plot_pre_olm_gates()
plt.show()